# Lecture: Anomaly Detection with GANs — f-AnoGAN

In notebook **C2-2** we detected anomalies with an **autoencoder**: train only on
normal data, then flag images with a high reconstruction error. This notebook
applies the *same idea* with a **GAN** instead of an autoencoder.

## AnoGAN → f-AnoGAN

**AnoGAN** (Schlegl et al., 2017) trains a GAN on normal images only. For a query
image $x$ it searches the latent code whose generated image best matches $x$:
$$z^* = \arg\min_z \; \| x - G(z) \|.$$
Normal images can be reproduced well (low residual); anomalies lie off the
learned manifold and cannot (high residual). The drawback: that search is an
**iterative optimisation per test image** — slow.

**f-AnoGAN** (Schlegl et al., 2019) fixes this by adding an **encoder** $E$ that
maps $x \to z$ in a single forward pass. The anomaly score combines an
image-space and a discriminator-feature residual:
$$A(x) = \| x - G(E(x)) \|^2 \;+\; \kappa \, \| f(x) - f(G(E(x))) \|^2$$
where $f(\cdot)$ are intermediate discriminator features.

So f-AnoGAN is essentially **an autoencoder whose decoder is a pretrained GAN
generator** — the direct bridge between the C2 autoencoder anomaly detector and
the C3 GANs. We use a model trained on the **MVTec-AD `bottle`** category
(defect-free bottles only) and score real *good* and *defective* test images.

Run the following cell only on Google Colab to copy the required
`fanogan.py` module into the working directory. Skip it when running locally.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C3-GANs/fanogan.py ./

## Loading the Pre-trained Models

f-AnoGAN has three components — generator `G`, discriminator `D`, encoder `E` —
all trained on defect-free bottle images. The checkpoints are committed under
`models/fanogan/`.

In [ ]:
import os
import torch
from fanogan import Generator, Discriminator, Encoder, anomaly_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LATENT_DIM = 128
CHANNELS   = 64

def resolve(path):
    for base in ("", "AIBIP/C3-GANs/"):
        if os.path.exists(base + path):
            return base + path
    return path

G = Generator(LATENT_DIM, CHANNELS).to(device)
D = Discriminator(CHANNELS).to(device)
E = Encoder(LATENT_DIM, CHANNELS).to(device)
G.load_state_dict(torch.load(resolve("models/fanogan/g.pth"), map_location=device))
D.load_state_dict(torch.load(resolve("models/fanogan/d.pth"), map_location=device))
E.load_state_dict(torch.load(resolve("models/fanogan/e.pth"), map_location=device))
G.eval(); D.eval(); E.eval()
print("Loaded G, D, E.")

## Demo Images

A small set of real MVTec bottle test images is shipped with the repository
(`fanogan_demo_assets/fanogan_demo.pt`): a few **good** bottles and a few
**defective** ones (cracks, contamination). No MVTec download required.

In [ ]:
demo_path = resolve("fanogan_demo_assets/fanogan_demo.pt")
demo = torch.load(demo_path, map_location=device)
images = demo["images"].to(device)   # (N, 3, 128, 128) in [-1, 1]
labels = demo["labels"]              # 0 = good, 1 = defective
names  = demo["names"]
print(f"{len(labels)} demo images:", names)

## 1) Anomaly Scores

We compute the f-AnoGAN score for every demo image. Good bottles should score
**low** (the generator reconstructs them well), defective ones **high**.

In [ ]:
import matplotlib.pyplot as plt

scores, residual = anomaly_score(images, G, D, E, kappa=1.0)
scores = scores.cpu()

colors = ["tab:green" if l == 0 else "tab:red" for l in labels]
plt.figure(figsize=(10, 4))
plt.bar(range(len(scores)), scores.numpy(), color=colors)
plt.xticks(range(len(names)), names, rotation=45, ha="right", fontsize=8)
plt.ylabel("anomaly score")
plt.title("f-AnoGAN anomaly scores (green = good, red = defective)")
plt.tight_layout()
plt.show()

## 2) Localising the Defect

The per-pixel residual $|x - G(E(x))|$ shows *where* the reconstruction fails.
For a good bottle it is near-uniform; for a defective one it lights up exactly
at the defect — a free segmentation map.

In [ ]:
def to_img(t):
    return ((t.cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()

with torch.no_grad():
    rec = G(E(images))

n = len(images)
fig, axes = plt.subplots(3, n, figsize=(2.2 * n, 6.5))
for i in range(n):
    axes[0, i].imshow(to_img(images[i]));           axes[0, i].axis("off")
    axes[0, i].set_title(names[i], fontsize=7)
    axes[1, i].imshow(to_img(rec[i]));              axes[1, i].axis("off")
    axes[2, i].imshow(residual[i, 0].cpu().numpy(), cmap="hot")
    axes[2, i].axis("off")
axes[0, 0].set_ylabel("input",     fontsize=9)
axes[1, 0].set_ylabel("G(E(x))",   fontsize=9)
axes[2, 0].set_ylabel("residual",  fontsize=9)
plt.tight_layout()
plt.show()

## 3) Thresholding — Good vs. Defective

Pick a threshold on the anomaly score that separates normal from anomalous. In
practice it is calibrated on a validation set of normal images (e.g. the mean
plus a few standard deviations of their scores).

In [ ]:
import numpy as np

good_scores = np.array([s for s, l in zip(scores.numpy(), labels) if l == 0])
threshold = good_scores.mean() + 2 * good_scores.std()
print(f"threshold = mean(good) + 2*std(good) = {threshold:.4f}")

for s, l, nm in zip(scores.numpy(), labels, names):
    pred = "DEFECTIVE" if s > threshold else "good"
    truth = "defective" if l == 1 else "good"
    mark = "ok " if (pred.lower().startswith(truth[:4])) else "MISS"
    print(f"  [{mark}] {nm:25s} score={s:7.4f} -> {pred}  (truth: {truth})")

---
## Reproducing the Weights

The checkpoints were produced on a GPU server with
[`train_fanogan.py`](train_fanogan.py), trained on the **defect-free** bottle
images only (two phases: WGAN-GP, then encoder):

```bash
python train_fanogan.py --data-dir bottle --epochs-gan 300 --epochs-enc 150
```

The MVTec-AD dataset is available from
[mvtec.com](https://www.mvtec.com/company/research/datasets/mvtec-ad) (one
category, e.g. `bottle`, is enough). The demo images were exported with
[`export_fanogan_demo.py`](export_fanogan_demo.py).

**Real-world use.** The same pipeline is used for industrial inspection (MVTec-AD
defect detection) and medical imaging (the original f-AnoGAN paper detects
anomalies in retinal OCT scans) — anywhere normal data is plentiful but labelled
anomalies are rare.